# MMAF-CD — Stage P1: slice-level pancreatic tumour dataset from MSD Task07

**Why:** the multi-cancer pancreatic subset gives every slice of a patient that patient's label (chest and pelvis slices are called "tumour"), has about 33 independent groups, and its specialist calls every healthy CT a tumour. MSD Task07 has voxel-level tumour masks, so each slice can be labelled correctly.

**What this notebook does (≈ 20–40 min, CPU is enough):**
1. streams `Task07_Pancreas.tar` (12.3 GB) from Hugging Face (fallback: the MONAI S3 mirror) — the tar is **never stored**, only one patient at a time
2. for each of the 281 labelled cases: up to **6 tumour slices** (tumour ≥ 50 px) and **3 tumour-free pancreas slices** (pancreas ≥ 200 px, no tumour), spread through the volume
3. CT window [−100, 240] HU → 8-bit PNG, 512×512, same window as the NIH slices used in Stage C1
4. writes `msd_pancreas_slices/{tumor,normal}/*.png` + `metadata.csv` (path, case, label, slice, pixel counts) + a sample grid

**Settings:** Accelerator **None (CPU)** is fine · **Internet ON** · no inputs needed.

**After it finishes:** Output tab → *New Dataset* → name it **`msd-pancreas-slices`** (public or private) → attach it to **E1 and E2**. They find it automatically.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# P1 — stream MSD Task07 Pancreas and extract slice-level labelled PNGs
# ════════════════════════════════════════════════════════════════════════════════
import os, io, sys, time, tarfile, shutil, gzip, subprocess
from pathlib import Path
import numpy as np, pandas as pd
try:
    import nibabel as nib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nibabel"], check=False); import nibabel as nib
import requests
from PIL import Image

URLS = ["https://huggingface.co/datasets/Novel-BioMedAI/Medical_Segmentation_Decathlon/resolve/main/Task07_Pancreas.tar",
        "https://msd-for-monai.s3-us-west-2.amazonaws.com/Task07_Pancreas.tar"]
OUT = Path("/kaggle/working/msd_pancreas_slices"); (OUT / "tumor").mkdir(parents=True, exist_ok=True); (OUT / "normal").mkdir(parents=True, exist_ok=True)
TMP = Path("/tmp/msd_pending"); TMP.mkdir(parents=True, exist_ok=True)     # holds an image or label until its partner arrives
N_TUMOR, N_NORMAL, MIN_TUMOR_PX, MIN_PANC_PX = 6, 3, 50, 200
HU_LO, HU_HI = -100.0, 240.0
T0 = time.time()

def spread(idx, k):
    """k indices spread evenly over idx (sorted)."""
    idx = sorted(int(i) for i in idx)
    if len(idx) <= k: return idx
    return [idx[int(round(j))] for j in np.linspace(0, len(idx) - 1, k)]

def to_png(sl):
    s = np.clip((sl.astype(np.float32) - HU_LO) / (HU_HI - HU_LO), 0, 1)
    return Image.fromarray((np.rot90(s) * 255).astype(np.uint8))

ROWS = []
def process(case, img_path, lab_path):
    img = nib.load(str(img_path)); lab = np.asarray(nib.load(str(lab_path)).dataobj)
    tum = (lab == 2).sum((0, 1)); pan = (lab >= 1).sum((0, 1))
    zt = spread(np.where(tum >= MIN_TUMOR_PX)[0], N_TUMOR)
    zn = spread(np.where((pan >= MIN_PANC_PX) & (tum == 0))[0], N_NORMAL)
    if not zt and not zn: return 0
    vol = np.asarray(img.dataobj)
    for z, y in [(z, "tumor") for z in zt] + [(z, "normal") for z in zn]:
        f = OUT / y / f"{case}_z{z:03d}.png"; to_png(vol[:, :, z]).save(f)
        ROWS.append({"path": f"{y}/{f.name}", "case": case, "label": y, "z": int(z), "n_slices": int(vol.shape[2]),
                     "tumor_px": int(tum[z]), "pancreas_px": int(pan[z])})
    return len(zt) + len(zn)

def stream(url):
    pending = {}                                     # case → {"img": path, "lab": path}
    n_cases = 0
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        r.raw.decode_content = True
        with tarfile.open(fileobj=r.raw, mode="r|*") as tf:
            for m in tf:
                name = m.name.replace("\\", "/")
                base = name.split("/")[-1]
                if not m.isfile() or base.startswith("._") or not base.endswith(".nii.gz"): continue
                kind = "img" if "/imagesTr/" in name else ("lab" if "/labelsTr/" in name else None)
                if kind is None: continue                # imagesTs has no labels
                case = base.replace(".nii.gz", "")
                p = TMP / f"{case}_{kind}.nii.gz"
                with tf.extractfile(m) as src, open(p, "wb") as dst: shutil.copyfileobj(src, dst, 1 << 20)
                pending.setdefault(case, {})[kind] = p
                if len(pending[case]) == 2:
                    try:
                        k = process(case, pending[case]["img"], pending[case]["lab"]); n_cases += 1
                        if n_cases % 20 == 0:
                            print(f"   {n_cases} cases · {len(ROWS)} slices · pending on disk {len(pending)-1} · {time.time()-T0:.0f}s", flush=True)
                    except Exception as e:
                        print(f"   ⚠️ {case}: {type(e).__name__}: {e}")
                    for q in pending.pop(case).values(): q.unlink(missing_ok=True)
    return n_cases

done = False
for url in URLS:
    try:
        print("streaming", url, flush=True); n = stream(url); print(f"✅ {n} labelled cases processed"); done = n > 0
        if done: break
    except Exception as e:
        print(f"   ⚠️ {url}: {type(e).__name__}: {e} — trying next source", flush=True)
        ROWS.clear()
        for d in ("tumor", "normal"):
            for f in (OUT / d).glob("*.png"): f.unlink()
assert done, "could not download MSD Task07 from any source"
shutil.rmtree(TMP, ignore_errors=True)

meta = pd.DataFrame(ROWS); meta.to_csv(OUT / "metadata.csv", index=False)
print(f"\nslices: {len(meta)}  ({meta.label.value_counts().to_dict()})  cases: {meta.case.nunique()}  "
      f"cases with tumour slices: {meta[meta.label=='tumor'].case.nunique()}  [{(time.time()-T0)/60:.1f} min]")

# sample grid for a visual check
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
rng = np.random.default_rng(0); fig, ax = plt.subplots(2, 8, figsize=(18, 5))
for r_, lab in enumerate(["tumor", "normal"]):
    sub = meta[meta.label == lab]
    for c_, i in enumerate(rng.choice(len(sub), min(8, len(sub)), replace=False)):
        row = sub.iloc[i]; ax[r_, c_].imshow(Image.open(OUT / row.path), cmap="gray")
        ax[r_, c_].set_title(f"{lab} {row.case} z{row.z}", fontsize=7); ax[r_, c_].axis("off")
plt.tight_layout(); plt.savefig(OUT / "samples.png", dpi=80); plt.close()
(OUT / "README.txt").write_text("MSD Task07 Pancreas — slice-level labels from voxel masks (label 2 = tumour).\n"
    f"tumour slice: >= {MIN_TUMOR_PX} tumour px (<= {N_TUMOR} per case); normal: >= {MIN_PANC_PX} pancreas px and no tumour (<= {N_NORMAL} per case).\n"
    f"window [{HU_LO}, {HU_HI}] HU, rot90. Group by 'case' when splitting.\n")
print("✅ output:", OUT, "→ Output tab → New Dataset → 'msd-pancreas-slices'")
